In [0]:
"""
Phase 2: Silver Layer - Fixed-Size Chunking
Splits raw 10-K text into fixed-size overlapping chunks for embedding.
Tagged with chunking_strategy="fixed_size_1000" to support later
comparison against section-aware chunking in MLflow.
"""

from pyspark.sql.functions import udf, explode, monotonically_increasing_id
from pyspark.sql.types import ArrayType, StringType

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

def fixed_size_chunk(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    if not text:
        return []
    chunks = []
    start = 0
    text_len = len(text)
    while start < text_len:
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunk_udf = udf(fixed_size_chunk, ArrayType(StringType()))

bronze_df = spark.table("rag_pipeline.main.bronze_10k_filings")

chunked_df = (
    bronze_df
    .withColumn("chunks", chunk_udf("raw_text"))
    .withColumn("chunk_text", explode("chunks"))
    .withColumn("chunk_id", monotonically_increasing_id())
    .withColumn("chunking_strategy", udf(lambda: "fixed_size_1000", StringType())())
    .select(
        "chunk_id",
        "company",
        "fiscal_year",
        "chunk_text",
        "chunking_strategy",
        "file_path"
    )
)

display(chunked_df.groupBy("company", "fiscal_year").count())

company,fiscal_year,count
MSFT,2024,460
AAPL,2024,252
META,2025,639
NVDA,2025,417
NVDA,2024,427
AAPL,2023,247
AAPL,2025,254
MSFT,2025,407
GOOG,2025,428
NVDA,2023,417


In [0]:
"""
Structure Aware Chunking
"""


import re
from pyspark.sql.functions import udf, explode, monotonically_increasing_id, lit
from pyspark.sql.types import ArrayType, StringType, StructType, StructField

# 10-K filings use standard "Item X" section headers
ITEM_PATTERN = re.compile(r'(Item\s+\d+[A-Za-z]?\.?\s)', re.IGNORECASE)

def section_aware_chunk(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    if not text:
        return []

    # Split on Item headers, keeping the header attached to its section
    splits = ITEM_PATTERN.split(text)
    sections = []
    current_section = ""
    for part in splits:
        if ITEM_PATTERN.match(part):
            if current_section:
                sections.append(current_section)
            current_section = part
        else:
            current_section += part
    if current_section:
        sections.append(current_section)

    # Sub-chunk any section that's still too long, using the same fixed-size logic
    chunks = []
    for section in sections:
        if len(section) <= chunk_size:
            chunks.append(section)
        else:
            start = 0
            while start < len(section):
                end = start + chunk_size
                chunks.append(section[start:end])
                start += chunk_size - overlap
    return chunks

section_chunk_udf = udf(section_aware_chunk, ArrayType(StringType()))

section_chunked_df = (
    bronze_df
    .withColumn("chunks", section_chunk_udf("raw_text"))
    .withColumn("chunk_text", explode("chunks"))
    .withColumn("chunk_id", monotonically_increasing_id())
    .withColumn("chunking_strategy", lit("section_aware"))
    .select(
        "chunk_id", "company", "fiscal_year",
        "chunk_text", "chunking_strategy", "file_path"
    )
)

display(section_chunked_df.groupBy("company", "fiscal_year").count())

company,fiscal_year,count
MSFT,2024,563
AAPL,2024,294
META,2025,674
NVDA,2025,466
NVDA,2024,474
AAPL,2023,287
AAPL,2025,294
MSFT,2025,503
GOOG,2025,487
NVDA,2023,463


In [0]:
"""
Structural comparison between the both strategies
"""

from pyspark.sql.functions import length, avg, min, max, count

fixed_stats = chunked_df.withColumn("len", length("chunk_text")) \
    .agg(count("*").alias("total_chunks"), avg("len").alias("avg_len"),
         min("len").alias("min_len"), max("len").alias("max_len")) \
    .withColumn("strategy", lit("fixed_size_1000"))

section_stats = section_chunked_df.withColumn("len", length("chunk_text")) \
    .agg(count("*").alias("total_chunks"), avg("len").alias("avg_len"),
         min("len").alias("min_len"), max("len").alias("max_len")) \
    .withColumn("strategy", lit("section_aware"))

display(fixed_stats.union(section_stats))

total_chunks,avg_len,min_len,max_len,strategy
6459,998.4801052794551,2,1000,fixed_size_1000
7321,870.2749624368256,4,1000,section_aware


In [0]:

"""46 chunks with less than 20 characters"""
display(
    section_chunked_df.filter(length("chunk_text") < 20)
    .select("company", "fiscal_year", "chunk_text", "chunking_strategy")
)

"""1 chunks with less than 20 characters"""
display(
    chunked_df.filter(length("chunk_text") < 20)
    .select("company", "fiscal_year", "chunk_text", "chunking_strategy")
)

company,fiscal_year,chunk_text,chunking_strategy
AAPL,2023,Item 1. Business 1,section_aware
AAPL,2024,Item 1. Business 1,section_aware
AAPL,2025,Item 1. Business 1,section_aware
GOOG,2023,Item 1. Business 4,section_aware
GOOG,2023,Item 8 as well as,section_aware
GOOG,2024,Item 1. Business 4,section_aware
GOOG,2024,"ss, see",section_aware
GOOG,2024,Item 8 as well as,section_aware
GOOG,2024,in,section_aware
GOOG,2025,Item 1. Business 3,section_aware


company,fiscal_year,chunk_text,chunking_strategy
META,2025,15,fixed_size_1000


In [0]:
MIN_CHUNK_LENGTH = 30

chunked_df_clean = chunked_df.filter(length("chunk_text") >= MIN_CHUNK_LENGTH)
section_chunked_df_clean = section_chunked_df.filter(length("chunk_text") >= MIN_CHUNK_LENGTH)


# write to delta tables
chunked_df_clean.write.format("delta").mode("overwrite").saveAsTable("rag_pipeline.main.silver_chunks_fixed_size")
section_chunked_df_clean.write.format("delta").mode("overwrite").saveAsTable("rag_pipeline.main.silver_chunks_section_aware")

print("Fixed-size chunks:", spark.table("rag_pipeline.main.silver_chunks_fixed_size").count())
print("Section-aware chunks:", spark.table("rag_pipeline.main.silver_chunks_section_aware").count())

Fixed-size chunks: 6457
Section-aware chunks: 7197
